# Running the whole network

`01_one_check.ipynb` drives a single check by hand. This one just lets the loop
run, over every reach, until you stop it or it runs out of things to do.

Two modes, one switch:

| `RUN_FOREVER` | behaviour |
|---|---|
| `False` | stop once there are no gaps and nothing is running |
| `True` | keep going, the way a service would; interrupt the cell to stop |

`True` is the honest one — the loop is not a job with an end, it is something
that keeps reality matching intent. `False` exists because a notebook that never
returns is awkward, and because a batch run wants to know when it is done.

Interrupting is always safe. Every fact the loop needs is in the database before
the step that wrote it returns, so stopping loses nothing — restart and it picks
up mid-flight jobs by asking which ones they are.

**Before running:** `docker compose up -d db minio minio-init`, a seeded network
(`01_one_check.ipynb` does that), and the `build_model` image available.

In [ ]:
import os
import time

import pandas as pd

from recon import check, db, jobs, processing, queue
from recon.config import settings
from recon.workers import LocalDockerRunner


def build_runner():
    """Jobs run as sibling containers, so they address services by name."""
    env_vars = {"AWS_ENDPOINT_URL": "http://minio:9000"}
    for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY",
                "AWS_SESSION_TOKEN", "AWS_REQUEST_PAYER"):
        if os.environ.get(key):
            env_vars[key] = os.environ[key]
    return LocalDockerRunner(
        image=settings.build_model_image,
        network=settings.docker_network,
        env_vars=env_vars,
        platform=settings.docker_platform,
        volumes=[f"{settings.docker_data_dir}:/data:ro"] if settings.docker_data_dir else [],
    )


def tally():
    return db.one("""
        SELECT count(*) FILTER (WHERE state = 'finished')  AS finished,
               count(*) FILTER (WHERE state = 'in_flight') AS in_flight,
               count(*) FILTER (WHERE state = 'resting')   AS resting,
               count(*) FILTER (WHERE state = 'halted')    AS halted,
               count(*)                                    AS total
        FROM reach_status""")


runner = build_runner()
print(f"image {runner.image} on network {runner.network}")
print(tally())

## Settings

`MAX_IN_FLIGHT` is the only one that matters much. The loop itself has no
opinion about how many jobs may run at once — that is still an open question in
`reconciliation-loop.md` — so until it is settled, the cap belongs to whoever is
running the thing rather than to the loop.

Note the cap is on *submissions*, not on checks. Checking a reach costs almost
nothing and cannot start work by itself, so a reach whose job is already running
still gets checked — that is how its result gets noticed.

In [ ]:
RUN_FOREVER = False   # True = keep running like a service; interrupt to stop
MAX_IN_FLIGHT = 4     # jobs allowed to run at once
INTERVAL = 8          # seconds between passes

## Where the network stands

`reach_status` derives each reach's state from the columns that already say it —
only `halted` is stored — so this cannot disagree with what the loop will do.

In [ ]:
print(pd.DataFrame(db.query(
    "SELECT state, count(*) AS reaches FROM reach_status GROUP BY state ORDER BY reaches DESC")))
print()
print(f"{len(queue.due_reaches())} reaches due a check")

## The loop

One pass is two questions put to the database, in this order:

1. **which jobs are we waiting on?** — poll them, and for any that finished,
   clear the marker and ask for a check. Done first so capacity freed by a
   finished job is usable in the same pass.
2. **which reaches are due?** — check each one: observe storage, work out the
   gap, act on it.

Neither question needs anything remembered from the last pass, which is why
interrupting this cell costs nothing.

In [ ]:
started = time.time()
passes = 0

try:
    while True:
        passes += 1

        for outcome in jobs.status_pass(runner):
            if outcome["status"] in ("succeeded", "failed"):
                print(f"    reach {outcome['reach_id']} {outcome['step']}: "
                      f"{outcome['status']} - {outcome['action']}")

        submitted = 0
        for row in queue.due_reaches():
            if len(processing.in_flight()) >= MAX_IN_FLIGHT:
                break
            if check.run_check(row["reach_id"], runner).submitted_ref:
                submitted += 1

        now = tally()
        print(f"[{time.time() - started:6.0f}s] pass {passes:3d}  "
              f"finished {now['finished']:>3}/{now['total']}  "
              f"in flight {now['in_flight']:>2}  resting {now['resting']:>2}  "
              f"halted {now['halted']:>2}  submitted {submitted}")

        settled = (now["in_flight"] == 0
                   and now["finished"] + now["halted"] == now["total"])
        if settled and not RUN_FOREVER:
            print(f"\nsettled after {passes} passes in {time.time() - started:.0f}s")
            break

        time.sleep(INTERVAL)

except KeyboardInterrupt:
    print("\nstopped by hand. Nothing is lost: jobs still running are recorded in")
    print("the database, and the next pass finds them by asking which they are.")

## What happened

In [ ]:
print(pd.DataFrame(db.query(
    "SELECT state, count(*) AS reaches FROM reach_status GROUP BY state ORDER BY reaches DESC")))
print()
print(db.one("""SELECT count(*) FILTER (WHERE applied_revision = desired_revision) AS caught_up,
                       count(*) FILTER (WHERE has_gap) AS with_gap,
                       count(DISTINCT model_id) AS models
                FROM reach_status"""))

## Anything parked

A halted reach failed enough times in a row that the loop stopped picking it up
and is asking for a person. Nothing else stops on its own — a reach that is
merely resting will be retried when its backoff expires.

In [ ]:
halted = db.query("""SELECT reach_id, consecutive_failures, halted_at, last_error
                     FROM reach_processing WHERE halted ORDER BY halted_at""")
if halted:
    display(pd.DataFrame(halted))
    print("\nafter fixing the cause:  processing.clear_halt(reach_id)")
else:
    print("nothing halted")

## Recent activity

In [ ]:
from recon import activity

pd.DataFrame(activity.recent(15))[["reach_id", "action", "outcome", "revision", "took"]]